# **Forecasting negative day-ahead electricity prices in Germany**

## Description

**Working title:** *Predicting negative day-ahead electricity prices in Germany: how much do machine learning models add over simple rules?*

Germany has a lot of solar and wind, so on sunny or windy low-demand hours the day-ahead price can drop below zero. That affects grid planning, battery storage, and how renewables are subsidized. The question is whether ML can predict those hours using only information available *before* the auction, and whether it beats a simple rule based on residual load (demand minus wind and solar).

**RQ1:** Do ML models (logistic regression, XGBoost) beat simple baselines at predicting negative-price hours a day ahead?
**RQ2:** Does performance hold over time as solar capacity grows, or do models degrade on newer years?

## Day 0-1: preparation

1. **Get data access.**
   - ENTSO-E Transparency Platform: register, then email their support to request a RESTful API token. It can take a few days, so do it first.
   - **Fallback with no token:** Energy-Charts (Fraunhofer ISE) and SMARD (Bundesnetzagentur) publish German price and generation data openly.
   - Use the `entsoe-py` library for downloads.
2. **Verify what the data looks like today.** The European day-ahead market recently changed its time resolution (from hourly to 15-minute products). Check what resolution your price series has over different years, and decide to aggregate everything to hourly. Write this in your methods section.
3. **Set up the repo and tools:** same structure as before (`data/`, `notebooks/`, `src/`, `results/`, `paper/`), plus pandas, scikit-learn, xgboost, shap, matplotlib, and `entsoe-py`.
4. **Set up paper tools:** a free Overleaf account with an arXiv-style or IEEE template, and Zotero for references.
5. **Email a potential supervisor or endorser** now, as before.

## Decisions

- **Market:** the German-Luxembourg bidding zone, day-ahead prices.
- **Target:** binary, "price < 0 EUR/MWh in this hour". Price level regression is a secondary extra.
- **No leakage!!!!!** The auction closes around midday the day before delivery. Use only features known by then:
  - Day-ahead forecasts of load, wind, and solar (published on ENTSO-E)
  - Calendar features: hour, weekday, month, holidays
  - Lagged prices up to the previous day
  - Do **not** use *actual* generation or *actual* weather for the delivery day. That information isn't available at forecast time and would make your results fake. Reviewers check this.
- **Derived feature:** residual load = forecast load - forecast wind - forecast solar.
- **Time-based splits, never random:**
  - Train: earliest years up to 2023
  - Validation: 2024
  - Test: 2025 to the latest full data you have, frozen and touched once
  - Note that 2022 was an energy-crisis year with abnormal prices. Treat it as a flagged period and test results with and without it.
- **Baselines:**
  - B1: same hour yesterday
  - B2: same hour last week
  - B3: threshold rule on residual load (pick the threshold on the training data)
- **Models:** logistic regression, then XGBoost.
- **Metrics:** negative hours are rare (a small percentage of hours), so accuracy is misleading. Use precision, recall, F1 for the negative class, PR-AUC, and Brier score. For the cost angle, define a simple asymmetric cost (missing a negative hour vs. a false alarm) and report results across a range of cost ratios instead of claiming one number.
- **Interpretation:** SHAP on the XGBoost model, to see whether it simply rediscovers residual load or finds something extra.

## Four-week plan

| Week | Tasks | Deliverables |
|---|---|---|
| **1** | Download data, clean it, align time zones and daylight-saving shifts, check missing values, exploratory plots, freeze the splits, run baselines B1-B3. Read 8-10 papers. | Clean dataset, baseline results, protocol committed |
| **2** | Logistic regression and XGBoost with time-series cross-validation, small hyperparameter tuning, test-set evaluation (once), bootstrap confidence intervals. | Main results table |
| **3** | Year-by-year performance analysis (RQ2), SHAP, error analysis (what do missed negative hours have in common?), cost-ratio analysis. | Figures and findings |
| **4** | Write the paper, a small dashboard (Streamlit or Dash) showing tomorrow's predicted negative hours, README, 2-minute demo video. | Paper draft, repo, demo |

**Tiers:** the must-have is baselines plus both models with one frozen test set. The should-have is the year-by-year analysis and SHAP. The nice-to-have is price-level regression, weather data, the dashboard polish, and a comparison with another country.

Watch out for daylight-saving time. Clock changes create 23-hour and 25-hour days, and they break joins silently if you don't handle them. Convert everything to UTC first.

## Paper preparation, running alongside

**Start in week 1:**
- **Literature search:** Google Scholar and arXiv with terms like "negative electricity prices forecasting", "day-ahead price forecasting Germany", and "renewable curtailment prediction". Save papers in Zotero and write 2-3 sentences per paper on what they did and what gap remains. Your gap is likely "few studies test how models hold up as solar capacity grows" (confirm this with your own search before you claim it).
- **Write the protocol** (I'll draft it, as before).

**Paper structure, filled in as you go:**
1. Abstract (write last)
2. Introduction: the problem, why negative prices matter, your contributions
3. Related work (from your reading notes)
4. Data and methods (mostly your protocol)
5. Results: table, year-by-year plot, SHAP plot
6. Discussion: what ML adds, when it fails, what it means for storage and grid planning
7. Limitations: one market, regime changes, forecast errors in the inputs, no weather modeling
8. Conclusion and future work (other countries, including Egypt's growing solar market)

**Where to publish:**
- **arXiv first.** It needs an endorsement for a new submitter, so ask your instructor or former professor early. Categories like cs.LG or eess.SY are likely fits.
- **Then workshops** on machine learning for climate or energy, and student or regional IEEE conferences. Look up current deadlines yourself, since they change every year.

## What this gives each audience

- **Recruiters:** time-series validation, leakage awareness, imbalanced classification, SHAP, and a deployed dashboard. These are skills energy, finance, and logistics companies hire for.
- **Masters committees:** a falsifiable question, honest evaluation, and a clear link to the energy transition, plus the option to frame a future thesis around Egypt or MENA solar integration.

## Next step

I can write the filled-in protocol for this project, the same way I did for the cybersecurity one, with every decision above turned into protocol text. Do you want me to do that next, or start with the data download and cleaning code?.
